In [14]:
import os
import numpy as np
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from scipy.optimize import minimize, NonlinearConstraint


# --- 2. Parameters ---
p = 8
T_lower_bound = 40.6
T1 = [50.0] * p
T2 = [30.0] * p
Tamb = [20.0] * p
Tmains = [60.0] * p
d = [2.5] * p

# --- 3. Objective Function ---
def objective(x):
    return sum(x)  # Minimize the sum of setpoints

# --- 4. Black-Box Nonlinear Constraint Function ---
def constraint_wrapper(x):
    from concurrent.futures import ThreadPoolExecutor
    max_workers = min(os.cpu_count() or 1, p)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Threads don't require pickling, so lambda functions work perfectly here!
        # We use a list comprehension to submit index-matched arguments to each thread
        futures = [
            executor.submit(predict_two_node, x[i], T1[i], T2[i], Tamb[i], Tmains[i], d[i])  #def predict_two_node(setpoint, temp_n1, temp_n2, temp_amb, temp_mains, draw, duration = 15, tank_volume = TANK_VOLUME):
            for i in range(p)
        ]
        
        # Gather the returned results from each thread in order
        results = [f.result() for f in futures]
        t_out_array = np.hstack(results)
    return t_out_array

# Define the constraint: T_out must be between 40.6 and 100
nl_constraint = NonlinearConstraint(constraint_wrapper, lb=T_lower_bound, ub=100)



In [16]:
# --- 5. Variable Bounds & Setup ---
bounds = [(49.0, 60.0) for _ in range(p)]
x0 = [55.0] * p

# --- 6. Solve Using a True Black-Box Method ---
result = minimize(
    objective,
    x0,
    method='cobyqa',  # Derivative-free trust-region SQP
    bounds=bounds,
    constraints=nl_constraint
)

print("Success:", result.success)
print("Optimized Setpoints:", np.round(result.x, 2))

TypeError: 'float' object is not subscriptable

In [8]:
# Requisites

import os
import datetime as dt
import pandas as pd
import numpy as np

from ochre import Dwelling
from ochre.utils import default_input_path  # for using sample files
from ochre import HeatPumpWaterHeater


#Global params

TANK_VOLUME = 151 #40g

deadband_default = 5.56  # in C


time_interval = 15 # adjust setpoint every 15 minutes?


#2 node simulation
def predict_two_node(setpoint, temp_n1, temp_n2, temp_amb, temp_mains, draw, duration = 15, tank_volume = TANK_VOLUME):
    setpoint_default = setpoint
    equipment_args = {
        "start_time": dt.datetime(2026, 1, 1, 0, 0), #10292, 90023,  # year, month, day, hour, minute
        "time_res": dt.timedelta(minutes=1),
        "duration": dt.timedelta(minutes = duration),
        "verbosity": 9,  # required to get setpoint and deadband in results
        "save_results": False,  # if True, must specify output_path
        # "output_path": os.getcwd(),        # Equipment parameters
        "Setpoint Temperature (C)": setpoint_default,
        "Tank Volume (L)": tank_volume,
        "Tank Height (m)": 1.22,
        "UA (W/K)": 2.17,
        "HPWH COP (-)": 4.5,
        "water_nodes": 2
    }

    # Create water draw schedule
    times = pd.date_range(
        equipment_args["start_time"],
        equipment_args["start_time"] + equipment_args["duration"],
        freq=equipment_args["time_res"],
        inclusive="left",
    )
    #withdraw_rate = np.random.choice([0, water_draw_magnitude], p=[0.99, 0.01], size=len(times))
    withdraw_rate = draw
    withdraw_rate = withdraw_rate[:len(times)]
    schedule = pd.DataFrame(
        {
            "Water Heating (L/min)": withdraw_rate,
            "Water Heating Setpoint (C)": setpoint_default,  # Setting so that it can reset
            "Water Heating Deadband (C)": deadband_default,  # Setting so that it can reset
            "Zone Temperature (C)": temp_amb,
            "Zone Wet Bulb Temperature (C)": 15,  # Required for HPWH
            "Mains Temperature (C)": temp_mains,
        },
        index=times,
    )

    # Initialize equipment
    hpwh = HeatPumpWaterHeater(schedule=schedule, **equipment_args)

    hpwh.model.states[:] = np.array([temp_n1, temp_n2])

    # Simulate
    data = pd.DataFrame()
    data = {'draw_data' :[], 'setpoint' :[]}
    control_signal = {}
    setpoints = []

    for t in hpwh.sim_times:
        # Change setpoint based on hour of day
        setpoint = setpoint_default
        control_signal = {
            "Setpoint": setpoint
        }

        setpoints.append(setpoint)
        # Run with controls
        _ = hpwh.update(control_signal=control_signal)

    
    df = hpwh.finalize()

    cols_to_save = [
        "Hot Water Outlet Temperature (C)",
        "T_WH1",
        "T_WH2"
    ]

    to_save = df.loc[:, cols_to_save]
    to_save = to_save[:-1]

    #return to_save
    return to_save["Hot Water Outlet Temperature (C)"]
